# Normal chain

In [11]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm= ChatOpenAI(model='gpt-4o-mini',temperature=0.6)
prompt=PromptTemplate.from_template('what is the definition of {word}')


output_parser=StrOutputParser()
chain = prompt | llm | output_parser
result= chain.invoke({'word': 'langchain'})
print (result)


LangChain is a framework designed for developing applications that utilize language models, particularly large language models (LLMs). It provides tools and components to streamline the process of building applications that can interact with these models in a more structured and effective way. LangChain can facilitate tasks such as chaining together various prompts, managing context, handling memory, and integrating with external data sources or APIs.

The framework is particularly useful for creating applications that require complex interactions with language models, such as chatbots, question-answering systems, and other natural language processing (NLP) applications. By offering abstractions for common tasks and patterns, LangChain helps developers leverage the capabilities of language models more efficiently and effectively.


# Sequential chain

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

llm= ChatOpenAI(model='gpt-4o-mini',temperature=0.6)
output_parser= StrOutputParser()

seq_template1=PromptTemplate.from_template("give me a summary on {topic}")

seq_template2= PromptTemplate.from_template("based on the summary, give me 5 key points \n {summary}")

seq_chain= seq_template1 | llm | output_parser | seq_template2 | llm | output_parser

seq_result = seq_chain.invoke({'topic':'langchain'})
print(seq_result)

Here are five key points about LangChain:

1. **Modular Design**: LangChain offers a variety of modular components that can be combined to create customized applications, including tools for prompt management, memory handling, and data source integration.

2. **Chain Operations**: The framework enables developers to create "chains" of operations, allowing for multi-step processes where the output of one step feeds into the next, enhancing workflow complexity.

3. **Stateful Interactions**: With built-in memory management, LangChain allows applications to maintain context across multiple user exchanges, facilitating more coherent and context-aware interactions.

4. **API Integration**: LangChain can connect with various APIs and external data sources, enabling LLMs to access real-time information, which improves the quality of responses generated by the models.

5. **Evaluation Tools**: The framework includes features for evaluating and testing language model performance, assisting deve

In [8]:
seq_chain.get_graph().print_ascii()

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +------------+       
      | ChatOpenAI |       
      +------------+       
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *       

# Parallel Chain

In [9]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.6)
parser= StrOutputParser()

par_template1=PromptTemplate.from_template("give me a summary on {topic}")

par_template2= PromptTemplate.from_template("based on the summary, give me 5 key points \n {topic}")

par_chain1= par_template1 | llm | parser
par_chain2= par_template2 | llm | parser

parallel_chain= RunnableParallel(
    {
        "summary":par_chain1,
        "key_points":par_chain2

    })

mer_template= PromptTemplate.from_template(""" combine these 2 in to comprehensive output:
                                            
                                            summary: {summary}
                                            key_point: {key_points}""")

mer_chain= mer_template | llm | parser

final_chain=  parallel_chain | mer_chain
result = final_chain.invoke({'topic':'llm'})
print( result)



**Summary of Large Language Models (LLMs)**

LLM stands for "Large Language Model," which is a type of artificial intelligence model designed to understand, generate, and manipulate human language. These models are trained on vast amounts of text data using machine learning techniques, particularly deep learning.

**Key Points about LLMs:**

1. **Architecture**: Many LLMs utilize transformer architecture, enabling efficient processing and generation of text by focusing on the relationships between words within sentences.

2. **Training**: LLMs undergo unsupervised learning on diverse datasets, including books, articles, and websites. This extensive training equips them with knowledge of grammar, factual information, reasoning abilities, and contextual understanding.

3. **Applications**: LLMs have a wide range of applications, including chatbots, virtual assistants, content generation, translation, summarization, and programming assistance.

4. **Examples**: Notable LLMs include OpenAI

In [12]:
final_chain.get_graph().print_ascii()

        +-----------------------------------+        
        | Parallel<summary,key_points>Input |        
        +-----------------------------------+        
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                    +------------+   
  | ChatOpenAI |                    | ChatOpenAI |   
  +------------+                    +------------+   
          *                               *          
          *                               *          
          *                               *          
+-----------------+         

# Conditional chain

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from langchain_core.runnables import RunnableParallel
from pydantic import BaseModel,Field
from typing import Literal
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model='gpt-4o-mini',temperature=0.6)
str_parser=StrOutputParser()

class ReviewSentiment(BaseModel):
    sentiment: Literal["positive","negative","neutral"]=Field(description="this is the sentiment of the movie review")

pydantic_parser=PydanticOutputParser(pydantic_object=ReviewSentiment)

con_temp1= PromptTemplate.from_template("""give me sentiment of the movie review {review} \n
                                        format_instructions: {format_instructions}""",
                                        partial_variables={'format_instructions':pydantic_parser.get_format_instructions()})
                                    
chain= con_temp1 | llm | pydantic_parser

print(chain.invoke({"review": "this is a bad movie, good quality"}))


sentiment='negative'


In [12]:
chain.get_graph().print_ascii()

    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
  +-----------------+    
  | ReviewSentiment |    
  +-----------------+    
